# PDF editing tools with `pypdf`

This notebook provides three small utilities:

1. Reorder pages
2. Delete selected pages
3. Merge PDFs

Page numbers are **1-based**. Ranges are inclusive, so `"2,4-6"` means pages 2, 4, 5, and 6.


In [ ]:
# Install once if needed
# %pip install pypdf

In [ ]:
from pathlib import Path
from typing import Iterable, List

from pypdf import PdfReader, PdfWriter


def parse_page_spec(spec: str, total_pages: int | None = None) -> List[int]:
    """
    Parse a 1-based page specification into 0-based page indices.

    Supports:
      "1,3,5"
      "1-4"
      "4-1"  # descending range
      "1,3-5,8"
    """
    if not spec or not spec.strip():
        raise ValueError("Page specification is empty.")

    result: List[int] = []
    parts = [part.strip() for part in spec.split(",") if part.strip()]

    for part in parts:
        if "-" in part:
            start_str, end_str = part.split("-", 1)
            start = int(start_str.strip())
            end = int(end_str.strip())
            step = 1 if end >= start else -1
            pages = range(start, end + step, step)
            result.extend(page - 1 for page in pages)
        else:
            result.append(int(part) - 1)

    if total_pages is not None:
        invalid = [idx + 1 for idx in result if idx < 0 or idx >= total_pages]
        if invalid:
            raise ValueError(
                f"Invalid page number(s): {invalid}. This PDF has {total_pages} page(s)."
            )

    return result


def ensure_parent_dir(path: str | Path) -> None:
    output = Path(path)
    if output.parent and not output.parent.exists():
        output.parent.mkdir(parents=True, exist_ok=True)


def read_pdf(path: str | Path, password: str | None = None) -> PdfReader:
    reader = PdfReader(str(path))
    if reader.is_encrypted:
        if password is None:
            raise ValueError(f"'{path}' is encrypted. Provide a password.")
        status = reader.decrypt(password)
        if status == 0:
            raise ValueError(f"Could not decrypt '{path}'. Check the password.")
    return reader


def write_pdf(writer: PdfWriter, output_path: str | Path) -> None:
    ensure_parent_dir(output_path)
    with open(output_path, "wb") as f:
        writer.write(f)


def copy_metadata(reader: PdfReader, writer: PdfWriter) -> None:
    if reader.metadata:
        metadata = {key: str(value) for key, value in reader.metadata.items() if value is not None}
        if metadata:
            writer.add_metadata(metadata)


## 1. Reorder pages

Example: `order="3,1-2,5"` makes the output pages `[3, 1, 2, 5]` from the original PDF.


In [ ]:
def reorder_pdf(
    input_path: str | Path,
    output_path: str | Path,
    order: str,
    password: str | None = None,
) -> None:
    reader = read_pdf(input_path, password=password)
    writer = PdfWriter()
    page_indices = parse_page_spec(order, total_pages=len(reader.pages))

    for idx in page_indices:
        writer.add_page(reader.pages[idx])

    copy_metadata(reader, writer)
    write_pdf(writer, output_path)
    print(f"Saved: {output_path}")


# Example
# reorder_pdf("input.pdf", "reordered.pdf", order="3,1-2,5")


## 2. Delete selected pages

Example: `pages_to_delete="2,4-6"` removes pages 2, 4, 5, and 6.


In [ ]:
def delete_pages_pdf(
    input_path: str | Path,
    output_path: str | Path,
    pages_to_delete: str,
    password: str | None = None,
) -> None:
    reader = read_pdf(input_path, password=password)
    writer = PdfWriter()
    delete_set = set(parse_page_spec(pages_to_delete, total_pages=len(reader.pages)))

    for idx, page in enumerate(reader.pages):
        if idx not in delete_set:
            writer.add_page(page)

    copy_metadata(reader, writer)
    write_pdf(writer, output_path)
    print(f"Saved: {output_path}")


# Example
# delete_pages_pdf("input.pdf", "deleted_pages.pdf", pages_to_delete="2,4-6")


## 3. Merge PDFs

The input list order is the merge order.


In [ ]:
def merge_pdfs(
    input_paths: Iterable[str | Path],
    output_path: str | Path,
    password: str | None = None,
) -> None:
    writer = PdfWriter()

    for input_path in input_paths:
        reader = read_pdf(input_path, password=password)
        for page in reader.pages:
            writer.add_page(page)

    write_pdf(writer, output_path)
    print(f"Saved: {output_path}")


# Example
# merge_pdfs(["part1.pdf", "part2.pdf", "part3.pdf"], "merged.pdf")


## Practical examples

Uncomment and modify these paths when you use the notebook.


In [ ]:
# reorder_pdf("/path/to/input.pdf", "/path/to/output_reordered.pdf", order="3,1-2,5")
# delete_pages_pdf("/path/to/input.pdf", "/path/to/output_deleted.pdf", pages_to_delete="2,4-6")
# merge_pdfs(["/path/to/a.pdf", "/path/to/b.pdf"], "/path/to/output_merged.pdf")
